импорт основных библиотек и датасета(продолжаем работать с датасетом о Пальмерских пингвинах)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import missingno as mgno
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import cross_val_score, train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix

будем импортировать уже прошедший EDA датасет

In [2]:
df = pd.read_csv("../hw02_EDA/Crak.csv")
df

,studyName,Sample Number,Species,Individual ID,Clutch Completion,Date Egg,Culmen Length (mm),Culmen Depth (mm),Flipper Length (mm),Body Mass (g),Sex,Delta 15 N (o/oo),Delta 13 C (o/oo),Island_Dream,Island_Torgersen,Culmen Length/Depth ratio,Flipper Length/Mass ratio
0,PAL0708,1,Adelie Penguin (Pygoscelis adeliae),N1A1,Yes,11/11/07,39.1,18.7,181.0,3750.0,0,8.733382,-25.686292,0,1,2.090909,0.048267
1,PAL0708,2,Adelie Penguin (Pygoscelis adeliae),N1A2,Yes,11/11/07,39.5,17.4,186.0,3800.0,1,8.949560,-24.694540,0,1,2.270115,0.048947
2,PAL0708,3,Adelie Penguin (Pygoscelis adeliae),N2A1,Yes,11/16/07,40.3,18.0,195.0,3250.0,1,8.368210,-25.333020,0,1,2.238889,0.060000
3,PAL0708,5,Adelie Penguin (Pygoscelis adeliae),N3A1,Yes,11/16/07,36.7,19.3,193.0,3450.0,1,8.766510,-25.324260,0,1,1.901554,0.055942
4,PAL0708,6,Adelie Penguin (Pygoscelis adeliae),N3A2,Yes,11/16/07,39.3,20.6,190.0,3650.0,0,8.664960,-25.298050,0,1,1.907767,0.052055
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
337,PAL0910,119,Gentoo penguin (Pygoscelis papua),N38A1,No,12/1/09,47.2,13.7,214.0,4925.0,1,7.991840,-26.205380,0,0,3.445255,0.043452
338,PAL0910,121,Gentoo penguin (Pygoscelis papua),N39A1,Yes,11/22/09,46.8,14.3,215.0,4850.0,1,8.411510,-26.138320,0,0,3.272727,0.044330
339,PAL0910,122,Gentoo penguin (Pygoscelis papua),N39A2,Yes,11/22/09,50.4,15.7,222.0,5750.0,0,8.301660,-26.041170,0,0,3.210191,0.038609
340,PAL0910,123,Gentoo penguin (Pygoscelis papua),N43A1,Yes,11/22/09,45.2,14.8,212.0,5200.0,1,8.242460,-26.119690,0,0,3.054054,0.040769


In [3]:
df.describe


<bound method NDFrame.describe of     studyName  Sample Number                              Species  \
0     PAL0708              1  Adelie Penguin (Pygoscelis adeliae)   
1     PAL0708              2  Adelie Penguin (Pygoscelis adeliae)   
2     PAL0708              3  Adelie Penguin (Pygoscelis adeliae)   
3     PAL0708              5  Adelie Penguin (Pygoscelis adeliae)   
4     PAL0708              6  Adelie Penguin (Pygoscelis adeliae)   
..        ...            ...                                  ...   
337   PAL0910            119    Gentoo penguin (Pygoscelis papua)   
338   PAL0910            121    Gentoo penguin (Pygoscelis papua)   
339   PAL0910            122    Gentoo penguin (Pygoscelis papua)   
340   PAL0910            123    Gentoo penguin (Pygoscelis papua)   
341   PAL0910            124    Gentoo penguin (Pygoscelis papua)   

    Individual ID Clutch Completion  Date Egg  Culmen Length (mm)  \
0            N1A1               Yes  11/11/07                39.1   

данные уже были плюс-минус причесаны и обработаны => нам остается данные нормализовать(для KNN это обязательно, т.к. из-за того что мы расссчитываем расстояние и одни признаки могут быть 10-30, а другие в диапазоне 3000-5000, и во втором случае просто расстояние будет слишком высоким и остальные признаки,имеющие меньшие порядки, не будут вносить сущест
    scвенный вклад в рассчет расстояния), также для дальнейшей классификации можно дропнуть столбцы с ID,Sample Number и Date Egg, и если мы хотим предсказать виднам не нужно применять OHE и нужно просто изначальные данные взять

импортируем заново датасет с пингвинами, только без OHE для видов пингвинов, так как мы будем их использовать в качестве таргета

теперь начнем с того что
a) выброшу ненужные столбцы
б)разбиение на train и test

In [4]:
df_clean = df.drop(columns = ["Individual ID","Sample Number","Date Egg","studyName"])
df_clean

,Species,Clutch Completion,Culmen Length (mm),Culmen Depth (mm),Flipper Length (mm),Body Mass (g),Sex,Delta 15 N (o/oo),Delta 13 C (o/oo),Island_Dream,Island_Torgersen,Culmen Length/Depth ratio,Flipper Length/Mass ratio
0,Adelie Penguin (Pygoscelis adeliae),Yes,39.1,18.7,181.0,3750.0,0,8.733382,-25.686292,0,1,2.090909,0.048267
1,Adelie Penguin (Pygoscelis adeliae),Yes,39.5,17.4,186.0,3800.0,1,8.949560,-24.694540,0,1,2.270115,0.048947
2,Adelie Penguin (Pygoscelis adeliae),Yes,40.3,18.0,195.0,3250.0,1,8.368210,-25.333020,0,1,2.238889,0.060000
3,Adelie Penguin (Pygoscelis adeliae),Yes,36.7,19.3,193.0,3450.0,1,8.766510,-25.324260,0,1,1.901554,0.055942
4,Adelie Penguin (Pygoscelis adeliae),Yes,39.3,20.6,190.0,3650.0,0,8.664960,-25.298050,0,1,1.907767,0.052055
...,...,...,...,...,...,...,...,...,...,...,...,...,...
337,Gentoo penguin (Pygoscelis papua),No,47.2,13.7,214.0,4925.0,1,7.991840,-26.205380,0,0,3.445255,0.043452
338,Gentoo penguin (Pygoscelis papua),Yes,46.8,14.3,215.0,4850.0,1,8.411510,-26.138320,0,0,3.272727,0.044330
339,Gentoo penguin (Pygoscelis papua),Yes,50.4,15.7,222.0,5750.0,0,8.301660,-26.041170,0,0,3.210191,0.038609
340,Gentoo penguin (Pygoscelis papua),Yes,45.2,14.8,212.0,5200.0,1,8.242460,-26.119690,0,0,3.054054,0.040769


также проведем OHE encoding для признака Clutch Completion

In [5]:
df_clean = pd.get_dummies(df_clean, columns = ["Clutch Completion"], dtype= int, drop_first= True)

In [6]:
X = df_clean.drop("Species", axis =1)
Y = df_clean["Species"]

x_train, x_test, y_train, y_test = train_test_split(X,Y, test_size = 0.25, random_state= 42)

In [15]:
knn_pipe = Pipeline([
    ('scaler', MinMaxScaler()),
    ('model',KNeighborsClassifier())    
])

param_grid = {
    "model__n_neighbors" : [3,5,7,9,11],
    "model__weights" :["distance","uniform"],
    "model__metric" : ["euclidean", 'manhattan',"minkowski","cosine"]
}

grid_search = GridSearchCV( 
    estimator = knn_pipe,
    param_grid= param_grid,
    scoring='accuracy',
    cv = 15,
    verbose = 0)

grid_search.fit(x_train,y_train)

print(grid_search.best_params_)
print(grid_search.best_score_)

{'model__metric': 'manhattan', 'model__n_neighbors': 5, 'model__weights': 'distance'}
0.996078431372549


впоспользовавшись GRidSEarch для поиска гиперпапаметров нашли оптимальные
{'model__metric': 'manhattan', 'model__n_neighbors': 5, 'model__weights': 'distance'}
0.996078431372549

ради интереса теперь попробуем без масштабирования

In [8]:
knn_model = KNeighborsClassifier( n_neighbors=5,
    weights='distance',
    metric='manhattan')


knn_model.fit(x_train,y_train)
y_pred = knn_model.predict(x_test)
accuracy_score(y_test,y_pred)

0.813953488372093

как мы можем увидеть без масштабирования точность сильно упала

In [17]:
Model = Pipeline([
    ('scaler', MinMaxScaler()),
    ('model',KNeighborsClassifier(n_neighbors= 5, weights= 'distance',metric='manhattan'))
])

Model.fit(x_train,y_train)
y_pred2 = Model.predict(x_test)



sc = classification_report(y_test,y_pred2)
cm = confusion_matrix(y_test,y_pred2)
print(cm)
print(sc)
print(accuracy_score(y_test,y_pred2))

[[43  1  0]
 [ 0 13  0]
 [ 0  0 29]]
                                           precision    recall  f1-score   support

      Adelie Penguin (Pygoscelis adeliae)       1.00      0.98      0.99        44
Chinstrap penguin (Pygoscelis antarctica)       0.93      1.00      0.96        13
        Gentoo penguin (Pygoscelis papua)       1.00      1.00      1.00        29

                                 accuracy                           0.99        86
                                macro avg       0.98      0.99      0.98        86
                             weighted avg       0.99      0.99      0.99        86

0.9883720930232558


из-за достаточно небольших размеров датасетта, и достаточно серьезных межвидовых отличий между пингвинами(облегчает задачу классификации в общем), KNN отлично подходит под установленную задачу(что доказывают высокие значения метрик), однако сам по себе алгоритм крайне затратные по памяти и времени, что будет негативно сказываться при гораздо более высоких данных, например если бы мы смогли собрать данные с  крупнонаселенного пингваними региона, то классифицировать их с помощью нашей модели было бы значительно более проблематично